# Quality factor delle risonanze (da un CSV)

Legge un CSV (per esempio `punto_NNN_....csv` con le colonne `CH1`, `CH2`), lo mostra, applica la **media mobile** (la stessa `moving_average` di `movemin.py`, `np.convolve` con `mode="same"`) e, **sulla curva della media mobile**, trova tutte le campane di risonanza visibili e ne calcola il quality factor:

**Q = altezza / larghezza**, con
- **altezza** = prominenza del picco, cioè l'altezza sopra la base locale (tra il picco e il minimo più alto dei due lati),
- **larghezza** = larghezza a metà altezza (FWHM) della prominenza, in campioni (oppure nelle unità di `COLONNA_X` se il CSV ha una colonna per l'asse x).

Le campane vengono cercate dentro la rampa di CH1 (se il CSV ha CH1 e `CANALE_RAMPA` è impostato) e senza i `window_length/2` campioni ai bordi, dove la media mobile con bordi a zero non è valida.

**Quante campane visualizzare:** `N_CAMPANE` in cella 1 (un numero, oppure `None` per tutte). Il calcolo e la tabella riguardano sempre **tutte** le campane trovate; l'ingrandimento di ciascuna è mostrato solo per le `N_CAMPANE` più prominenti.

Ordine: cella 1 (scrivi `PERCORSO_CSV` e scegli `N_CAMPANE`), 2 (visualizza il CSV), 3 (media mobile e ricerca delle campane), 4 (tabella con Q), 5 (grafici).

In [ ]:
# Cella 1 - impostazioni
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks, peak_widths

PERCORSO_CSV = ""          # <-- SCRIVI QUI il file CSV, es. "2026-10-06/run_007/punto_011_x91.818_y146.526.csv" (relativo al notebook) o percorso completo
N_CAMPANE = 3              # <-- quante campane visualizzare in dettaglio (le piu' prominenti); None = tutte quelle trovate

CANALE = "CH2"             # colonna da analizzare
CANALE_RAMPA = "CH1"       # colonna usata per limitare la ricerca alla rampa; None = tutto il record
COLONNA_X = None           # colonna per l'asse x (es. lunghezza d'onda); None = indice del campione
CAMPANE_VERSO_ALTO = True  # True: risonanze = massimi; False: risonanze = minimi (la curva viene invertita per la ricerca)

WINDOW_LENGTH = 10000      # lunghezza della media mobile in campioni (come window_length di movemin.py)
SOGLIA_PROMINENZA = 0.1    # una campana conta se la sua altezza e' almeno questa frazione dell'escursione max-min nella zona
DISTANZA_MIN = None        # distanza minima tra due campane in campioni; None = WINDOW_LENGTH
FRAZIONE_ALTEZZA = 0.5     # a che frazione dell'altezza si misura la larghezza (0.5 = larghezza a meta' altezza, FWHM)

V_INIZIO_V = 0.9           # rampa di CH1: inizio quando CH1 supera questo valore
TOLLERANZA_FINE_V = 0.3    # rampa di CH1: fine quando CH1 e' a meno di questo dal suo massimo


def moving_average(values, window_length):
    """Media mobile con finestra di lunghezza variabile."""   # identica a movemin.py
    if window_length <= 0:
        raise ValueError('window_length deve essere > 0')

    window_length = int(window_length)
    if window_length % 2 == 0:
        window_length += 1

    kernel = np.ones(window_length) / window_length
    return np.convolve(values, kernel, mode='same')


def rampa_ch1(ch1):
    """(inizio, fine) della rampa in salita di CH1, oppure None."""
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    i1 = i0 + int(np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)[0])
    return (i0, i1) if i1 > i0 else None

In [ ]:
# Cella 2 - legge e visualizza il CSV
if not PERCORSO_CSV.strip():
    raise ValueError("Scrivi in cella 1 il file CSV in PERCORSO_CSV")
percorso = os.path.abspath(PERCORSO_CSV)
df = pd.read_csv(percorso)
print(f"File: {percorso}")
print(f"{len(df)} righe, colonne: {list(df.columns)}")
if CANALE not in df.columns:
    raise ValueError(f"Colonna {CANALE!r} non trovata: colonne disponibili {list(df.columns)}")

x_tutto = df[COLONNA_X].to_numpy(dtype=float) if COLONNA_X else np.arange(len(df), dtype=float)
etichetta_x = COLONNA_X if COLONNA_X else "indice del campione"

colonne_dati = [c for c in df.columns if c != COLONNA_X]
passo = max(1, len(df) // 20000)   # solo per il disegno
fig, assi = plt.subplots(len(colonne_dati), 1, figsize=(10, 2.5 * len(colonne_dati) + 1), sharex=True, squeeze=False)
for ax, nome in zip(assi[:, 0], colonne_dati):
    ax.plot(x_tutto[::passo], df[nome].to_numpy()[::passo], lw=0.8)
    ax.set_ylabel(nome)
assi[-1, 0].set_xlabel(etichetta_x)
assi[0, 0].set_title(os.path.basename(percorso))
plt.tight_layout()
plt.show()

In [ ]:
# Cella 3 - media mobile e ricerca di tutte le campane (sulla curva della media mobile)
y = df[CANALE].to_numpy(dtype=float)
liscia = moving_average(y, WINDOW_LENGTH)

mezza = (int(WINDOW_LENGTH) + (int(WINDOW_LENGTH) % 2 == 0)) // 2
a, b = mezza, len(y) - 1 - mezza            # via i bordi dove la media con bordi a zero non e' valida
if CANALE_RAMPA and CANALE_RAMPA in df.columns:
    rampa = rampa_ch1(df[CANALE_RAMPA].to_numpy(dtype=float))
    if rampa is None:
        raise RuntimeError(f"Nessuna rampa in {CANALE_RAMPA} (mai sopra {V_INIZIO_V} V)")
    a, b = max(a, rampa[0]), min(b, rampa[1])
    print(f"Zona di ricerca: rampa di {CANALE_RAMPA}, campioni {a}..{b}")
else:
    print(f"Zona di ricerca: tutto il record senza i bordi, campioni {a}..{b}")
if b <= a:
    raise RuntimeError("Zona di ricerca vuota")

zona = liscia[a:b + 1]
segno = 1.0 if CAMPANE_VERSO_ALTO else -1.0
escursione = zona.max() - zona.min()
picchi, prop = find_peaks(
    segno * zona,
    prominence=SOGLIA_PROMINENZA * escursione,
    distance=int(DISTANZA_MIN if DISTANZA_MIN else WINDOW_LENGTH),
)
print(f"Campane trovate: {len(picchi)} (soglia: altezza >= {SOGLIA_PROMINENZA * escursione:.4g} sulla curva liscia)")

campane = []
if len(picchi):
    larghezze, altezza_tagli, sinistra, destra = peak_widths(segno * zona, picchi, rel_height=FRAZIONE_ALTEZZA, prominence_data=(prop["prominences"], prop["left_bases"], prop["right_bases"]))
    for k, p in enumerate(picchi):
        idx = a + int(p)
        # posizioni (in campioni) dei due punti a meta' altezza, poi nell'asse x scelto
        xl = np.interp(a + sinistra[k], np.arange(len(y)), x_tutto)
        xr = np.interp(a + destra[k], np.arange(len(y)), x_tutto)
        larghezza = float(xr - xl)
        altezza = float(prop["prominences"][k])
        campane.append({
            "n": 0, "indice": idx, "x": float(x_tutto[idx]), "valore": float(liscia[idx]),
            "altezza": altezza, "larghezza": larghezza, "Q": altezza / larghezza,
            "xl": float(xl), "xr": float(xr),
            "livello_larghezza": float(segno * altezza_tagli[k]),
            "sinistra_idx": a + float(sinistra[k]), "destra_idx": a + float(destra[k]),
        })
    for n, c in enumerate(sorted(campane, key=lambda c: c["x"]), start=1):
        c["n"] = n

In [ ]:
# Cella 4 - tabella con il quality factor di TUTTE le campane trovate
unita_x = COLONNA_X if COLONNA_X else "campioni"
print(f"{'n':>2} {'posizione':>14} {'altezza (V)':>12} {'larghezza (' + unita_x + ')':>26} {'Q = altezza/larghezza':>24}")
for c in sorted(campane, key=lambda c: c["x"]):
    print(f"{c['n']:>2} {c['x']:>14.6g} {c['altezza']:>12.4g} {c['larghezza']:>26.6g} {c['Q']:>24.4g}")
if not campane:
    print("(nessuna campana: abbassa SOGLIA_PROMINENZA o controlla CANALE_RAMPA e WINDOW_LENGTH)")

In [ ]:
# Cella 5 - grafici: panoramica con tutte le campane e ingrandimento delle N_CAMPANE piu' prominenti
passo = max(1, len(y) // 20000)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(x_tutto[::passo], y[::passo], color="gray", lw=0.6, alpha=0.6, label=f"{CANALE} grezzo")
ax.plot(x_tutto[::passo], liscia[::passo], color="red", lw=1.5, label=f"media mobile (N={WINDOW_LENGTH})")
ax.axvspan(x_tutto[a], x_tutto[b], color="orange", alpha=0.12, label="zona di ricerca")
for c in campane:
    ax.plot(c["x"], c["valore"], "kv", ms=7)
    ax.annotate(str(c["n"]), (c["x"], c["valore"]), textcoords="offset points", xytext=(0, 8), ha="center")
ax.set_xlabel(etichetta_x)
ax.set_ylabel(CANALE)
ax.set_title(f"{len(campane)} campane trovate")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

da_mostrare = sorted(campane, key=lambda c: -c["altezza"])
if N_CAMPANE is not None:
    da_mostrare = da_mostrare[:int(N_CAMPANE)]
da_mostrare = sorted(da_mostrare, key=lambda c: c["x"])

if da_mostrare:
    fig, assi = plt.subplots(1, len(da_mostrare), figsize=(4.2 * len(da_mostrare), 3.8), squeeze=False)
    for ax, c in zip(assi[0], da_mostrare):
        mezzo = max(2.5 * c["larghezza"], 1e-12)
        sel = (x_tutto >= c["x"] - mezzo) & (x_tutto <= c["x"] + mezzo)
        passo_z = max(1, int(sel.sum()) // 3000)
        ax.plot(x_tutto[sel][::passo_z], y[sel][::passo_z], color="gray", lw=0.6, alpha=0.6)
        ax.plot(x_tutto[sel][::passo_z], liscia[sel][::passo_z], color="red", lw=1.5)
        ax.hlines(c["livello_larghezza"], c["xl"], c["xr"], color="blue", lw=2, label="larghezza")
        base = c["valore"] - segno * c["altezza"]   # sotto il picco (campane verso l'alto) o sopra (verso il basso)
        ax.vlines(c["x"], min(base, c["valore"]), max(base, c["valore"]), color="green", lw=2, label="altezza")
        ax.set_title(f"campana {c['n']}: Q = {c['Q']:.3g}", fontsize=10)
        ax.set_xlabel(etichetta_x)
    assi[0, 0].legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()